# مسئلهٔ ۱ — V2 / ارزیابی A2 روی MP4 کامل با Sliding Window

این نوت‌بوک مدل برتر فعلی A2 را روی MP4 کامل اجرا می‌کند. ویدئو به پنجره‌های پنج‌ثانیه‌ای با stride برابر ۲٫۵ ثانیه تقسیم می‌شود، هر پنجره ۱۶ فریم دارد و در پایان احتمال پنجره‌ها به یک تصمیم video-level تجمیع می‌شود.

زمان رخداد فقط برای گزارش تحلیلی ثبت می‌شود و هرگز به مدل داده نمی‌شود. بنابراین همین pipeline برای MP4 جدیدِ بدون label نیز قابل استفاده است.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
import torch
from torch import nn
from torchvision import models
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
MODEL_DIR = DATA_ROOT / 'models_v2'
A2_CHECKPOINT_PATH = MODEL_DIR / 'resnet18_meanmax_pooling_frozen_best.pt'
INFERENCE_DIR = DATA_ROOT / 'inference_v2'
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)
WINDOW_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_validation_sliding_window_predictions.csv'
VIDEO_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_validation_sliding_video_predictions.csv'
AGGREGATION_RESULTS_PATH = INFERENCE_DIR / 'a2_validation_aggregation_comparison.csv'
THRESHOLD_CURVES_PATH = INFERENCE_DIR / 'a2_validation_aggregation_threshold_curves.csv'
METRICS_PATH = INFERENCE_DIR / 'a2_validation_sliding_metrics.json'

WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 2.5
NUM_FRAMES = 16
TARGET_HEIGHT = 224
TARGET_WIDTH = 320
FEATURE_DIM = 512
REUSE_CACHED_WINDOW_PREDICTIONS = True

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert VIDEO_MANIFEST_PATH.exists(), 'Run notebook 07 first.'
assert A2_CHECKPOINT_PATH.exists(), 'Run notebook 11 first.'
print(f'Device: {device}')

Device: cpu


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
video_manifest = pd.read_csv(VIDEO_MANIFEST_PATH).copy()
video_manifest['video_id'] = video_manifest['video_id'].astype(str)
video_manifest['label'] = video_manifest['label'].astype(int)
video_manifest['time_of_event'] = pd.to_numeric(video_manifest['time_of_event'], errors='coerce')
video_manifest['duration'] = pd.to_numeric(video_manifest['duration'], errors='coerce')
video_manifest['is_valid'] = video_manifest['is_valid'].astype(str).str.lower().eq('true')
validation_videos = video_manifest.loc[video_manifest['split'].eq('validation') & video_manifest['is_valid']].copy()
validation_videos = validation_videos.sort_values('video_id', key=lambda values: values.astype(int)).reset_index(drop=True)

assert len(validation_videos) == 120
assert validation_videos.groupby('label').size().to_dict() == {0: 60, 1: 60}
assert validation_videos['video_path'].map(lambda value: Path(value).is_file()).all()
display(pd.crosstab(validation_videos['split'], validation_videos['label']))

class ResNet18MeanMaxPoolingHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(FEATURE_DIM * 2),
            nn.Dropout(0.35),
            nn.Linear(FEATURE_DIM * 2, 1),
        )

    def forward(self, sequence_features: torch.Tensor) -> torch.Tensor:
        mean_features = sequence_features.mean(dim=1)
        max_features = sequence_features.max(dim=1).values
        return self.classifier(torch.cat([mean_features, max_features], dim=1)).squeeze(1)

weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone = models.resnet18(weights=weights)
encoder = nn.Sequential(*list(backbone.children())[:-1]).to(device).eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
head = ResNet18MeanMaxPoolingHead().to(device).eval()
checkpoint = torch.load(A2_CHECKPOINT_PATH, map_location=device, weights_only=False)
head.load_state_dict(checkpoint['model_state_dict'])
print({'A2_checkpoint_epoch': checkpoint['epoch'], 'A2_validation_PR_AUC': checkpoint['validation_pr_auc']})

label,0,1
split,,
validation,60,60


{'A2_checkpoint_epoch': 25, 'A2_validation_PR_AUC': 0.742437236181591}


In [3]:
def resize_letterbox_rgb(image_rgb: np.ndarray) -> np.ndarray:
    height, width = image_rgb.shape[:2]
    scale = min(TARGET_WIDTH / width, TARGET_HEIGHT / height)
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(image_rgb, (new_width, new_height), interpolation=interpolation)
    pad_x, pad_y = TARGET_WIDTH - new_width, TARGET_HEIGHT - new_height
    left, right = pad_x // 2, pad_x - pad_x // 2
    top, bottom = pad_y // 2, pad_y - pad_y // 2
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_REPLICATE)

def normalized_tensor(image_rgb: np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(image_rgb.copy()).permute(2, 0, 1).float().div_(255.0)
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def window_starts(duration: float) -> list[float]:
    if duration < WINDOW_SECONDS:
        return []
    starts = list(np.arange(0.0, duration - WINDOW_SECONDS + 1e-8, WINDOW_STRIDE_SECONDS))
    final_start = duration - WINDOW_SECONDS
    if not starts or not np.isclose(starts[-1], final_start):
        starts.append(final_start)
    return sorted({round(float(start), 6) for start in starts})

def decode_rgb_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float):
    step = 1.0 / fps if np.isfinite(fps) and fps > 0 else 1.0 / 30.0
    for attempt, offset in enumerate((0.0, step, -step, 2 * step)):
        target_timestamp = max(0.0, timestamp + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, target_timestamp * 1000.0)
        ok, image_bgr = cap.read()
        if ok and image_bgr is not None:
            status = 'exact' if attempt == 0 else f'seek_fallback_{attempt}'
            return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), status
    return None, 'decode_failed'

@torch.inference_mode()
def score_video_windows(video_row: pd.Series) -> list[dict]:
    duration = float(video_row.duration)
    starts = window_starts(duration)
    cap = cv2.VideoCapture(str(video_row.video_path))
    if not cap.isOpened():
        return [{'video_id': video_row.video_id, 'label': int(video_row.label), 'video_path': video_row.video_path,
                 'duration': duration, 'time_of_event': video_row.time_of_event, 'window_start': np.nan,
                 'window_end': np.nan, 'window_center': np.nan, 'positive_probability': np.nan,
                 'valid_frames': 0, 'decode_status': 'cannot_open', 'event_in_window': False}]

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    records = []
    try:
        for start in starts:
            end = start + WINDOW_SECONDS
            timestamps = np.linspace(start, end, num=NUM_FRAMES, endpoint=False)
            frames, statuses, previous_frame = [], [], None
            for timestamp in timestamps:
                frame_rgb, status = decode_rgb_at_timestamp(cap, float(timestamp), fps)
                if frame_rgb is None and previous_frame is not None:
                    frame_rgb, status = previous_frame.copy(), 'repeated_previous_after_decode_failure'
                if frame_rgb is not None:
                    previous_frame = frame_rgb
                    frames.append(normalized_tensor(resize_letterbox_rgb(frame_rgb)))
                statuses.append(status)

            probability = np.nan
            if len(frames) == NUM_FRAMES:
                image_tensor = torch.stack(frames).to(device)
                sequence_features = encoder(image_tensor).flatten(1).unsqueeze(0)
                probability = float(torch.sigmoid(head(sequence_features)).item())

            event_time = video_row.time_of_event
            event_in_window = bool(pd.notna(event_time) and start <= float(event_time) <= end)
            records.append({
                'video_id': video_row.video_id, 'label': int(video_row.label), 'video_path': video_row.video_path,
                'duration': duration, 'time_of_event': event_time, 'window_start': start, 'window_end': end,
                'window_center': start + WINDOW_SECONDS / 2, 'positive_probability': probability,
                'valid_frames': len(frames), 'decode_status': ';'.join(sorted(set(statuses))),
                'event_in_window': event_in_window,
            })
    finally:
        cap.release()
    return records

## اجرای sliding-window روی ۱۲۰ MP4 validation

اجرای نخست به‌علت decode و ResNet روی تمام پنجره‌ها زمان‌بر است. پس از ساخته‌شدن CSV پیش‌بینی پنجره‌ها، اجرای دوباره آن را reuse می‌کند و فقط aggregation و threshold را دوباره محاسبه می‌کند.

In [4]:
expected_video_ids = set(validation_videos['video_id'])
reuse_cache = False
if REUSE_CACHED_WINDOW_PREDICTIONS and WINDOW_PREDICTIONS_PATH.exists():
    cached_windows = pd.read_csv(WINDOW_PREDICTIONS_PATH)
    cached_windows['video_id'] = cached_windows['video_id'].astype(str)
    same_config = {'window_start', 'window_end', 'positive_probability', 'valid_frames'}.issubset(cached_windows.columns)
    reuse_cache = same_config and set(cached_windows['video_id']) == expected_video_ids

if reuse_cache:
    window_predictions = cached_windows
    print(f'Reused cached window predictions: {WINDOW_PREDICTIONS_PATH}')
else:
    records = []
    for _, video_row in tqdm(validation_videos.iterrows(), total=len(validation_videos), desc='Sliding-window validation'):
        records.extend(score_video_windows(video_row))
    window_predictions = pd.DataFrame(records)
    window_predictions.to_csv(WINDOW_PREDICTIONS_PATH, index=False)
    print(f'Window predictions saved: {WINDOW_PREDICTIONS_PATH}')

window_predictions['video_id'] = window_predictions['video_id'].astype(str)
assert set(window_predictions['video_id']) == expected_video_ids
assert window_predictions['positive_probability'].notna().all(), 'Some windows could not be decoded; inspect decode_status.'
assert window_predictions['valid_frames'].eq(NUM_FRAMES).all(), 'Some windows have an invalid frame mask.'
print({'window_rows': len(window_predictions), 'videos': window_predictions['video_id'].nunique()})
display(window_predictions['decode_status'].value_counts().to_frame('windows'))

Sliding-window validation: 100%|██████████| 120/120 [3:21:25<00:00, 100.71s/it] 

Window predictions saved: P:\NexarCollisionData\inference_v2\a2_validation_sliding_window_predictions.csv
{'window_rows': 1768, 'videos': 120}


,windows
decode_status,
exact,1768


In [5]:
def aggregate_probabilities(probabilities: np.ndarray) -> dict:
    ordered = np.sort(probabilities)[::-1]
    return {
        'max': float(ordered[0]),
        'top2_mean': float(ordered[:min(2, len(ordered))].mean()),
        'top3_mean': float(ordered[:min(3, len(ordered))].mean()),
        'mean': float(ordered.mean()),
    }

def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

video_records = []
for _, video_row in validation_videos.iterrows():
    video_windows = window_predictions.loc[window_predictions['video_id'].eq(video_row.video_id)].copy()
    probabilities = video_windows['positive_probability'].to_numpy(dtype=float)
    best_window = video_windows.iloc[int(np.argmax(probabilities))]
    record = {
        'video_id': video_row.video_id, 'label': int(video_row.label), 'video_path': video_row.video_path,
        'time_of_event': video_row.time_of_event, 'duration': float(video_row.duration),
        'num_windows': len(video_windows), 'max_window_start': float(best_window.window_start),
        'max_window_end': float(best_window.window_end), 'max_window_center': float(best_window.window_center),
    }
    record.update(aggregate_probabilities(probabilities))
    video_records.append(record)
video_predictions = pd.DataFrame(video_records)

aggregation_rows, threshold_rows = [], []
for aggregation_name in ('max', 'top2_mean', 'top3_mean', 'mean'):
    probabilities = video_predictions[aggregation_name].to_numpy(dtype=float)
    labels = video_predictions['label'].to_numpy(dtype=int)
    curve = pd.DataFrame([
        {'aggregation': aggregation_name, **binary_metrics(labels, probabilities, float(threshold))}
        for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2)
    ])
    threshold_rows.append(curve)
    best = curve.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0].to_dict()
    aggregation_rows.append({'aggregation': aggregation_name, **best})

aggregation_results = pd.DataFrame(aggregation_rows).sort_values(['f1', 'recall', 'precision'], ascending=False).reset_index(drop=True)
threshold_curves = pd.concat(threshold_rows, ignore_index=True)
selected = aggregation_results.iloc[0]
selected_aggregation = str(selected['aggregation'])
selected_threshold = float(selected['threshold'])
video_predictions['selected_aggregation'] = selected_aggregation
video_predictions['video_probability'] = video_predictions[selected_aggregation]
video_predictions['selected_threshold'] = selected_threshold
video_predictions['prediction'] = (video_predictions['video_probability'] >= selected_threshold).astype(int)

VIDEO_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
video_predictions.to_csv(VIDEO_PREDICTIONS_PATH, index=False)
aggregation_results.to_csv(AGGREGATION_RESULTS_PATH, index=False)
threshold_curves.to_csv(THRESHOLD_CURVES_PATH, index=False)

metrics_payload = {
    'model': 'A2 ResNet18 frozen + mean-max pooling',
    'evaluation_scope': 'full-MP4 validation with sliding windows; time_of_event not used for prediction',
    'window_seconds': WINDOW_SECONDS,
    'window_stride_seconds': WINDOW_STRIDE_SECONDS,
    'frames_per_window': NUM_FRAMES,
    'selected_aggregation': selected_aggregation,
    'selected_threshold_by_validation_f1': selected_threshold,
    'selected_metrics': {key: selected[key] for key in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'confusion_matrix']},
    'videos': int(len(video_predictions)),
    'windows': int(len(window_predictions)),
}
METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

print('Aggregation comparison:')
display(aggregation_results)
print(f'Selected aggregation: {selected_aggregation}; threshold: {selected_threshold:.2f}')
print(metrics_payload['selected_metrics'])

Aggregation comparison:


,aggregation,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,confusion_matrix
0,max,0.57,0.691667,0.641975,0.866667,0.737589,0.723889,0.708070,"[[31, 29], [8, 52]]"
1,top2_mean,0.47,0.658333,0.604396,0.916667,0.728477,0.726389,0.721350,"[[24, 36], [5, 55]]"
2,top3_mean,0.60,0.700000,0.671429,0.783333,0.723077,0.724444,0.726051,"[[37, 23], [13, 47]]"
3,mean,0.30,0.641667,0.600000,0.850000,0.703448,0.706944,0.708256,"[[26, 34], [9, 51]]"


Selected aggregation: max; threshold: 0.57
{'accuracy': np.float64(0.6916666666666667), 'precision': np.float64(0.6419753086419753), 'recall': np.float64(0.8666666666666667), 'f1': np.float64(0.7375886524822695), 'roc_auc': np.float64(0.7238888888888889), 'pr_auc': np.float64(0.7080697126972808), 'confusion_matrix': [[31, 29], [8, 52]]}


In [6]:
def predict_full_mp4(video_path: str | Path, aggregation: str = selected_aggregation, threshold: float = selected_threshold) -> dict:
    """Reusable inference for any valid MP4; no label or event time is required."""
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open MP4: {video_path}')
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = frame_count / fps if fps > 0 else np.nan
    if not np.isfinite(duration) or duration < WINDOW_SECONDS:
        raise ValueError(f'Video must be at least {WINDOW_SECONDS} seconds long.')

    unknown_video = pd.Series({
        'video_id': video_path.stem, 'label': 0, 'video_path': str(video_path),
        'duration': duration, 'time_of_event': np.nan,
    })
    records = score_video_windows(unknown_video)
    probabilities = np.asarray([record['positive_probability'] for record in records], dtype=float)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('At least one inference window could not be decoded.')
    video_probability = aggregate_probabilities(probabilities)[aggregation]
    best_record = records[int(np.argmax(probabilities))]
    return {
        'video_path': str(video_path), 'video_probability': float(video_probability),
        'prediction': int(video_probability >= threshold), 'threshold': float(threshold),
        'aggregation': aggregation, 'num_windows': len(records),
        'highest_probability_window': [best_record['window_start'], best_record['window_end']],
        'window_predictions': records,
    }

print('Reusable function predict_full_mp4(...) is ready for a new MP4.')
print(f'Video-level predictions: {VIDEO_PREDICTIONS_PATH}')
print(f'Metrics: {METRICS_PATH}')

Reusable function predict_full_mp4(...) is ready for a new MP4.
Video-level predictions: P:\NexarCollisionData\inference_v2\a2_validation_sliding_video_predictions.csv
Metrics: P:\NexarCollisionData\inference_v2\a2_validation_sliding_metrics.json
